In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ECommerce-Fraud-Detection")
    .master("local[*]")
    .getOrCreate()
)
# Ẩn các thông báo log rác, chỉ hiển thị thông báo lỗi hệ thống
spark.sparkContext.setLogLevel("ERROR")

In [2]:
# Đường dẫn lưu trữ tập dữ liệu trên hệ thống tệp tin phân tán HDFS
hdfs_path = "hdfs://localhost:9000/ecommerce-fraud-detection/transactions.csv"
# Đọc dữ liệu từ HDFS vào Spark DataFrame
df = spark.read.csv(
    hdfs_path,
    header=True,
    inferSchema=True
)
dataframe = df.limit(5).toPandas()

In [3]:
df.createOrReplaceTempView("ecommerce_transactions")
spark.catalog.listTables()

[Table(name='ecommerce_transactions', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [4]:
query_1 = spark.sql("""
    SELECT
        CASE
            WHEN is_fraud = 1 THEN 'Fraudulent'
            ELSE 'Legitimate'
        END AS transaction_type,
        COUNT(*) AS transaction_count,
        CAST(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER() AS DECIMAL(5, 2)) AS percentage,
        SUM(amount) AS total_amount,
        AVG(amount) AS average_amount
    FROM ecommerce_transactions
    GROUP BY is_fraud;
""")

query_1.toPandas().style.hide(axis="index")

transaction_type,transaction_count,percentage,total_amount,average_amount
Fraudulent,6612,2.21,3907435.450000,590.961199
Legitimate,293083,97.79,49188112.710000,167.829976


In [5]:
query_2 = spark.sql("""
    SELECT
        merchant_category,
        COUNT(*) AS fraud_count,
        SUM(amount) AS total_loss
    FROM ecommerce_transactions
    WHERE is_fraud = 1
    GROUP BY merchant_category
    ORDER BY total_loss DESC
""")
query_2.toPandas().style.hide(axis="index")

merchant_category,fraud_count,total_loss
electronics,1357,840480.360000
travel,1388,786641.000000
fashion,1340,775658.030000
gaming,1295,761831.010000
grocery,1232,742825.050000


In [6]:
query_3 = spark.sql("""
    SELECT
        HOUR(transaction_time) AS hour_of_day,
        country,
        SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) AS fraud_count,
        COUNT(*) AS total_transactions,
        CAST(SUM(CASE WHEN is_fraud = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*) AS DECIMAL(5, 2)) AS fraud_rate
    FROM ecommerce_transactions
    WHERE country IS NOT NULL AND country != 'Unknown'
    GROUP BY hour_of_day, country
    ORDER BY fraud_count DESC
    LIMIT 10;
""")

query_3.toPandas().style.hide(axis="index")

hour_of_day,country,fraud_count,total_transactions,fraud_rate
23,TR,47,1213,3.87
10,TR,43,1264,3.40
8,US,42,1349,3.11
11,TR,42,1231,3.41
19,GB,41,1320,3.11
7,TR,41,1258,3.26
17,PL,40,1208,3.31
16,TR,40,1197,3.34
9,TR,40,1155,3.46
11,PL,40,1291,3.10


In [7]:
query_4 = spark.sql("""
WITH ChannelStats AS (
    SELECT channel,
        COUNT(*) AS total_transactions,
        SUM(is_fraud) AS fraud_transactions,
        ROUND(SUM(is_fraud) * 100.0 / COUNT(*), 2) AS fraud_rate,
        AVG(amount) AS average_amount
    FROM ecommerce_transactions
    GROUP BY channel
)

SELECT
    RANK() OVER (ORDER BY fraud_rate DESC) AS ranking,
    channel,
    total_transactions,
    fraud_transactions,
    fraud_rate,
    average_amount
FROM ChannelStats
ORDER BY ranking
""")

query_4.toPandas().style.hide(axis="index")

ranking,channel,total_transactions,fraud_transactions,fraud_rate,average_amount
1,web,152226,5426,3.56,182.670108
2,app,147469,1186,0.80,171.482876


In [8]:
query_5 = spark.sql("""
SELECT
    CASE
        WHEN shipping_distance_km <= 50 THEN '0-50 km'
        WHEN shipping_distance_km <= 500 THEN '51-500 km'
        WHEN shipping_distance_km <= 2000 THEN '501-2000 km'
        ELSE '>2000 km'
    END AS shipping_group,

    COUNT(*) AS total_transactions,
    SUM(is_fraud) AS fraud_transactions,
    ROUND(SUM(is_fraud) * 100.0 / COUNT(*), 2) AS fraud_rate,
    AVG(amount) AS average_amount
FROM ecommerce_transactions

GROUP BY
    CASE
        WHEN shipping_distance_km <= 50 THEN '0-50 km'
        WHEN shipping_distance_km <= 500 THEN '51-500 km'
        WHEN shipping_distance_km <= 2000 THEN '501-2000 km'
        ELSE '>2000 km'
    END
ORDER BY fraud_rate DESC
""")

query_5.toPandas().style.hide(axis="index")

shipping_group,total_transactions,fraud_transactions,fraud_rate,average_amount
>2000 km,6864,1535,22.36,263.122190
501-2000 km,18167,2443,13.45,222.258582
51-500 km,247239,2384,0.96,171.891223
0-50 km,27425,250,0.91,173.326956


In [9]:
query_6 = spark.sql("""
    SELECT
        CASE
            WHEN account_age_days <= 30 THEN '0-30 days (New Account)'
            WHEN account_age_days > 30 AND account_age_days <= 180 THEN '31-180 days (Relatively New)'
            WHEN account_age_days > 180 AND account_age_days <= 365 THEN '181-365 days (Relatively Old)'
            ELSE '> 365 days (Old Account)'
        END AS account_age_group,

        COUNT(*) AS total_transactions,
        SUM(is_fraud) AS fraud_transactions,
        CAST(SUM(is_fraud) * 100.0 / COUNT(*) AS DECIMAL(5, 2)) AS fraud_rate
    FROM ecommerce_transactions
    GROUP BY account_age_group
    ORDER BY fraud_rate DESC;
""")

query_6.toPandas().style.hide(axis="index")

account_age_group,total_transactions,fraud_transactions,fraud_rate
0-30 days (New Account),2081,1051,50.50
31-180 days (Relatively New),17512,2205,12.59
> 365 days (Old Account),250488,3008,1.20
181-365 days (Relatively Old),29614,348,1.18


In [10]:
query_7 = spark.sql("""
SELECT
    DATE_FORMAT(transaction_time, 'EEEE') AS day_of_week,
    CASE
        WHEN amount < 50 THEN 'Under 50 USD'
        WHEN amount < 200 THEN '50 - 200 USD'
        WHEN amount < 500 THEN '200 - 500 USD'
        WHEN amount < 1000 THEN '500 - 1000 USD'
        ELSE 'Over 1000 USD'
    END AS amount_group,

    COUNT(*) AS total_transactions,
    SUM(is_fraud) AS fraud_transactions,
    ROUND(SUM(is_fraud) * 100.0 / COUNT(*), 2) AS fraud_rate,
    AVG(amount) AS average_amount
FROM ecommerce_transactions
GROUP BY day_of_week, amount_group
ORDER BY MIN(DAYOFWEEK(transaction_time)) ASC,
         MIN(amount) DESC,
         fraud_rate DESC, fraud_transactions DESC
""")

query_7.toPandas().style.hide(axis="index")


day_of_week,amount_group,total_transactions,fraud_transactions,fraud_rate,average_amount
Sunday,Over 1000 USD,895,254,28.38,1765.445609
Sunday,500 - 1000 USD,1922,32,1.66,679.089875
Sunday,200 - 500 USD,7085,76,1.07,303.370563
Sunday,50 - 200 USD,19546,245,1.25,106.372534
Sunday,Under 50 USD,12662,328,2.59,27.672248
Monday,Over 1000 USD,939,244,25.99,1714.778807
Monday,500 - 1000 USD,1963,26,1.32,676.724646
Monday,200 - 500 USD,7369,95,1.29,305.686499
Monday,50 - 200 USD,19691,266,1.35,106.488223
Monday,Under 50 USD,12973,329,2.54,27.723038


In [11]:
query_8 = spark.sql("""
    WITH AccountBehaviorStats AS (
        SELECT is_fraud,
            CASE
                WHEN (SUM(is_fraud) OVER (PARTITION BY user_id) - is_fraud) > 0 THEN 'Has Fraud History'
                ELSE 'Clean History'
            END AS history_status
        FROM ecommerce_transactions
    ),

    FraudSummary AS (
        SELECT history_status,
            COUNT(*) AS total_transactions,
            SUM(is_fraud) AS fraud_transactions
        FROM AccountBehaviorStats
        GROUP BY history_status
    )

    SELECT history_status,total_transactions, fraud_transactions,
        ROUND(fraud_transactions * 100.0 / total_transactions, 2) AS fraud_rate,
        ROUND(fraud_transactions * 100.0 / SUM(fraud_transactions) OVER(), 2) AS fraud_contribution_rate
    FROM FraudSummary
    ORDER BY fraud_rate DESC;
""")

query_8.toPandas().style.hide(axis="index")

history_status,total_transactions,fraud_transactions,fraud_rate,fraud_contribution_rate
Has Fraud History,93031,6040,6.49,91.35
Clean History,206664,572,0.28,8.65


In [12]:
query_9 = spark.sql("""
    SELECT
        CASE
            WHEN amount > (avg_amount_user * 5) THEN 'Abnormal (Amount > 5 * Avg)'
            ELSE 'Normal'
        END AS spending_type,
        COUNT(*) AS total_transactions,
        SUM(is_fraud) AS fraud_transactions,
        CAST(SUM(is_fraud) * 100.0 / COUNT(*) AS DECIMAL(5, 2)) AS fraud_rate
    FROM ecommerce_transactions
    WHERE avg_amount_user > 0
    GROUP BY spending_type;
""")

query_9.toPandas().style.hide(axis="index")


spending_type,total_transactions,fraud_transactions,fraud_rate
Abnormal (Amount > 5 * Avg),1673,1503,89.84
Normal,298022,5109,1.71


In [13]:
query_10 = spark.sql("""
    SELECT
        CASE
            WHEN country = bin_country THEN 'Matched'
            ELSE 'Mismatched'
        END AS geographic_status,

        COUNT(*) AS total_transactions,
        SUM(is_fraud) AS fraud_transactions,
        CAST(SUM(is_fraud) * 100.0 / COUNT(*) AS DECIMAL(5, 2)) AS fraud_rate
    FROM ecommerce_transactions
    WHERE
        country IS NOT NULL AND bin_country IS NOT NULL
        AND country != 'Unknown' AND bin_country != 'Unknown'
    GROUP BY geographic_status;
""")

query_10.toPandas().style.hide(axis="index")

geographic_status,total_transactions,fraud_transactions,fraud_rate
Matched,275965,3936,1.43
Mismatched,23730,2676,11.28


In [14]:
query_11 = spark.sql("""
    SELECT
        CASE
            WHEN promo_used = 1 THEN 'Promo Used'
            ELSE 'No Promo'
        END AS promotion_status,
        COUNT(*) AS total_transactions,
        SUM(is_fraud) AS fraud_transactions,
        ROUND(SUM(is_fraud) * 100.0 / COUNT(*), 2) AS fraud_rate,
        AVG(amount) AS average_amount
    FROM ecommerce_transactions
    GROUP BY promotion_status
    ORDER BY fraud_rate DESC;
""")

query_11.toPandas().style.hide(axis="index")

promotion_status,total_transactions,fraud_transactions,fraud_rate,average_amount
Promo Used,46045,2085,4.53,187.101807
No Promo,253650,4527,1.78,175.361504


In [15]:
query_12 = spark.sql("""
    WITH FraudStats AS (
        SELECT
            CASE
                WHEN cvv_result = 1 THEN 'Correct CVV'
                WHEN cvv_result = 0 THEN 'Incorrect CVV'
                ELSE 'Unknown'
            END AS cvv_result,

            CASE
                WHEN three_ds_flag = 1 THEN '3DS Enabled'
                ELSE '3DS Disabled'
            END AS three_ds_flag,

            is_fraud
        FROM ecommerce_transactions
    )

    SELECT
        cvv_result,
        three_ds_flag,
        COUNT(*) AS total_transactions,
        SUM(is_fraud) AS fraud_transactions,
        CAST(SUM(is_fraud) * 100.0 / COUNT(*) AS DECIMAL(5, 2)) AS fraud_rate
    FROM FraudStats
    GROUP BY cvv_result, three_ds_flag
    ORDER BY cvv_result, three_ds_flag;
""")

query_12.toPandas().style.hide(axis="index")

cvv_result,three_ds_flag,total_transactions,fraud_transactions,fraud_rate
Correct CVV,3DS Disabled,36988,698,1.89
Correct CVV,3DS Enabled,224379,1849,0.82
Incorrect CVV,3DS Disabled,27570,3661,13.28
Incorrect CVV,3DS Enabled,10758,404,3.76


In [16]:
query_13 = spark.sql("""
WITH Geo_Spatial_Stats AS (
    SELECT
        CASE WHEN avs_match = 1 THEN 'Address Matched' ELSE 'Address Mismatched' END AS avs_status,
        CASE WHEN country = bin_country THEN 'Same Country' ELSE 'Different Country' END AS geographic_status,
        CASE
            WHEN shipping_distance_km <= 10 THEN '1. Short (<=10km)'
            WHEN shipping_distance_km <= 50 THEN '2. Medium (10-50km)'
            WHEN shipping_distance_km <= 200 THEN '3. Long (50-200km)'
            ELSE '4. Very Long (>200km)'
        END AS delivery_range,
        is_fraud
    FROM ecommerce_transactions
)

SELECT
    avs_status,
    geographic_status,
    delivery_range,
    COUNT(*) AS total_transactions,
    COUNT(CASE WHEN is_fraud = 1 THEN 1 ELSE NULL END) AS fraud_transactions,
    CAST((COUNT(CASE WHEN is_fraud = 1 THEN 1 ELSE NULL END) / COUNT(*)) * 100 AS DECIMAL(5,2)) AS fraud_rate
FROM Geo_Spatial_Stats
GROUP BY avs_status, geographic_status, delivery_range
ORDER BY avs_status ASC, geographic_status DESC, delivery_range ASC;
""")

query_13.toPandas().style.hide(axis="index")

avs_status,geographic_status,delivery_range,total_transactions,fraud_transactions,fraud_rate
Address Matched,Same Country,1. Short (<=10km),4638,12,0.26
Address Matched,Same Country,2. Medium (10-50km),18522,56,0.30
Address Matched,Same Country,3. Long (50-200km),69786,240,0.34
Address Matched,Same Country,4. Very Long (>200km),139475,822,0.59
Address Matched,Different Country,4. Very Long (>200km),18723,789,4.21
Address Mismatched,Same Country,1. Short (<=10km),809,37,4.57
Address Mismatched,Same Country,2. Medium (10-50km),3456,145,4.20
Address Mismatched,Same Country,3. Long (50-200km),12722,582,4.57
Address Mismatched,Same Country,4. Very Long (>200km),26557,2042,7.69
Address Mismatched,Different Country,4. Very Long (>200km),5007,1887,37.69


In [17]:
query_14 = spark.sql("""
WITH UserBehavior AS (
    SELECT
        user_id,
        COUNT(*) AS total_transactions,
        SUM(is_fraud) AS fraud_transactions,
        SUM(CASE WHEN cvv_result = 0 THEN 1 ELSE 0 END) AS failed_cvv_attempts,
        SUM(CASE WHEN country != bin_country THEN 1 ELSE 0 END) AS geographic_mismatch
    FROM ecommerce_transactions
    GROUP BY user_id
),

RiskScored AS (
    SELECT *,
    (
       ((CAST(fraud_transactions AS DOUBLE) / CAST(total_transactions AS DOUBLE)) * 0.0221)
       + (failed_cvv_attempts * 0.0454)
       + (geographic_mismatch * 0.0264)
    ) AS risk_score
    FROM UserBehavior
)

SELECT
    DENSE_RANK() OVER (ORDER BY risk_score DESC) AS ranking,
    user_id,
    fraud_transactions,
    failed_cvv_attempts,
    geographic_mismatch,
    risk_score
FROM RiskScored
ORDER BY risk_score DESC
LIMIT 10;
""")

query_14.toPandas().style.hide(axis="index")

ranking,user_id,fraud_transactions,failed_cvv_attempts,geographic_mismatch,risk_score
1,1093,33,25,20,1.676023
2,5918,33,25,19,1.648755
3,1801,30,26,16,1.613850
4,5058,34,27,14,1.608136
5,5754,32,24,18,1.576993
6,291,30,25,15,1.542237
7,2572,33,26,13,1.536174
8,3140,31,24,16,1.524234
9,4449,30,23,16,1.479350
10,4873,30,23,16,1.478878
